In [1]:
import json
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, BooleanType
)
from pyspark.sql.functions import col, from_json, to_json, coalesce

In [12]:
spark.sql("DROP TABLE IF EXISTS demo.bronze.customers_cdc")
spark.sql("DROP TABLE IF EXISTS demo.bronze.drivers_cdc")

DataFrame[]

In [13]:
spark = (
    SparkSession.builder
    .appName("bronze-cdc")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
            "org.postgresql:postgresql:42.7.3",
            "org.apache.hadoop:hadoop-aws:3.3.4",
        ])
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.demo.type", "rest")
    .config("spark.sql.catalog.demo.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.demo.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")

    # IMPORTANT: Iceberg AWS/MinIO settings
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.demo.s3.path-style-access", "true")
    .config("spark.sql.catalog.demo.s3.access-key-id", "admin")
    .config("spark.sql.catalog.demo.s3.secret-access-key", "admin123")
    .config("spark.sql.catalog.demo.client.region", "us-east-1")

    # keep Hadoop settings too
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [14]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.gold")

DataFrame[]

In [15]:
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.customers_cdc (
    pk_id BIGINT,
    topic STRING,
    kafka_partition INT,
    kafka_offset BIGINT,
    kafka_timestamp TIMESTAMP,
    op STRING,
    lsn BIGINT,
    event_ts_ms BIGINT,
    before_json STRING,
    after_json STRING
)
USING iceberg
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.drivers_cdc (
    pk_id BIGINT,
    topic STRING,
    kafka_partition INT,
    kafka_offset BIGINT,
    kafka_timestamp TIMESTAMP,
    op STRING,
    lsn BIGINT,
    event_ts_ms BIGINT,
    before_json STRING,
    after_json STRING
)
USING iceberg
""")

DataFrame[]

In [16]:
def get_starting_offsets(bronze_table: str, topic_name: str) -> str:
    try:
        df = spark.sql(f"""
            SELECT kafka_partition, MAX(kafka_offset) AS max_offset
            FROM {bronze_table}
            GROUP BY kafka_partition
        """)
        rows = df.collect()
        if not rows:
            return "earliest"

        offsets = {
            topic_name: {
                str(r["kafka_partition"]): int(r["max_offset"]) + 1
                for r in rows
            }
        }
        return json.dumps(offsets)
    except Exception:
        return "earliest"


customers_offsets = get_starting_offsets(
    "demo.bronze.customers_cdc",
    "dbserver1.public.customers"
)

drivers_offsets = get_starting_offsets(
    "demo.bronze.drivers_cdc",
    "dbserver1.public.drivers"
)

print("customers startingOffsets =", customers_offsets)
print("drivers startingOffsets =", drivers_offsets)

customers startingOffsets = earliest
drivers startingOffsets = earliest


In [17]:
customers_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", customers_offsets)
    .option("endingOffsets", "latest")
    .load()
    .select(
        col("topic"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("timestamp").alias("kafka_timestamp"),
        col("key").cast("string").alias("key"),
        col("value").cast("string").alias("value"),
    )
)

drivers_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "dbserver1.public.drivers")
    .option("startingOffsets", drivers_offsets)
    .option("endingOffsets", "latest")
    .load()
    .select(
        col("topic"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("timestamp").alias("kafka_timestamp"),
        col("key").cast("string").alias("key"),
        col("value").cast("string").alias("value"),
    )
)

print("customers raw:", customers_raw.count())
print("drivers raw:", drivers_raw.count())

customers raw: 26263
drivers raw: 17151


In [18]:
customers_tombstones = customers_raw.filter(col("value").isNull())
drivers_tombstones = drivers_raw.filter(col("value").isNull())

print("customer tombstones:", customers_tombstones.count())
print("driver tombstones:", drivers_tombstones.count())

customers_raw = customers_raw.filter(col("value").isNotNull())
drivers_raw = drivers_raw.filter(col("value").isNotNull())

customer tombstones: 4352
driver tombstones: 2749


In [19]:
customer_schema = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("created_at", LongType(), True),
    StructField("updated_at", LongType(), True),
])

driver_schema = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("license_number", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("rating", DoubleType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("created_at", LongType(), True),
    StructField("updated_at", LongType(), True),
])

def make_envelope_schema(row_schema):
    return StructType([
        StructField("payload", StructType([
            StructField("before", row_schema, True),
            StructField("after", row_schema, True),
            StructField("source", StructType([
                StructField("lsn", LongType(), True),
            ]), True),
            StructField("op", StringType(), True),
            StructField("ts_ms", LongType(), True),
        ]), True)
    ])

def parse_debezium(df, row_schema):
    envelope_schema = make_envelope_schema(row_schema)

    parsed = df.withColumn("json_data", from_json(col("value"), envelope_schema))

    return parsed.select(
        col("topic"),
        col("kafka_partition"),
        col("kafka_offset"),
        col("kafka_timestamp"),
        col("json_data.payload.op").alias("op"),
        col("json_data.payload.source.lsn").alias("lsn"),
        col("json_data.payload.ts_ms").alias("event_ts_ms"),
        coalesce(
            col("json_data.payload.after.id"),
            col("json_data.payload.before.id")
        ).alias("pk_id"),
        to_json(col("json_data.payload.before")).alias("before_json"),
        to_json(col("json_data.payload.after")).alias("after_json"),
    )

customers_bronze = parse_debezium(customers_raw, customer_schema)
drivers_bronze = parse_debezium(drivers_raw, driver_schema)

In [20]:
customers_bronze.show(10, truncate=False)
drivers_bronze.show(10, truncate=False)

+--------------------------+---------------+------------+-----------------------+---+--------+-------------+-----+-----------+-----------------------------------------------------------------------------------------------------+
|topic                     |kafka_partition|kafka_offset|kafka_timestamp        |op |lsn     |event_ts_ms  |pk_id|before_json|after_json                                                                                           |
+--------------------------+---------------+------------+-----------------------+---+--------+-------------+-----+-----------+-----------------------------------------------------------------------------------------------------+
|dbserver1.public.customers|0              |0           |2026-04-19 14:08:36.989|r  |27752320|1776607716552|82   |NULL       |{"id":82,"name":"Lars Park","email":"updated_82_599@inbox.org","created_at":1776605647491169}        |
|dbserver1.public.customers|0              |1           |2026-04-19 14:08:36.989|r  

In [21]:
customers_bronze.writeTo("demo.bronze.customers_cdc").append()
drivers_bronze.writeTo("demo.bronze.drivers_cdc").append()

In [22]:
spark.sql("""
SELECT op, COUNT(*) AS n
FROM demo.bronze.customers_cdc
GROUP BY op
ORDER BY op
""").show()

spark.sql("""
SELECT op, COUNT(*) AS n
FROM demo.bronze.drivers_cdc
GROUP BY op
ORDER BY op
""").show()

+---+----+
| op|   n|
+---+----+
|  c|8645|
|  d|4352|
|  r| 241|
|  u|8673|
+---+----+

+---+----+
| op|   n|
+---+----+
|  c|4336|
|  d|2749|
|  r|  76|
|  u|7241|
+---+----+



In [23]:
spark.sql("""
SELECT topic, kafka_partition, kafka_offset, COUNT(*) AS n
FROM demo.bronze.customers_cdc
GROUP BY topic, kafka_partition, kafka_offset
HAVING COUNT(*) > 1
""").show()

spark.sql("""
SELECT topic, kafka_partition, kafka_offset, COUNT(*) AS n
FROM demo.bronze.drivers_cdc
GROUP BY topic, kafka_partition, kafka_offset
HAVING COUNT(*) > 1
""").show()

+-----+---------------+------------+---+
|topic|kafka_partition|kafka_offset|  n|
+-----+---------------+------------+---+
+-----+---------------+------------+---+

+-----+---------------+------------+---+
|topic|kafka_partition|kafka_offset|  n|
+-----+---------------+------------+---+
+-----+---------------+------------+---+

